# 中間表現の最適化

このノートブックでは、`qret opt` を使って中間表現（IR）を最適化する手順を確認します。
`qret opt` の実行方法は次の 2 つです。

- `--pass` を並べて直接実行する。
- YAML パイプラインで pass と外部コマンドをまとめて実行する。

本章の目標は次のとおりです。

- `Call` のインライン展開（1 段・再帰）を比較する。
- `ir::decompose_inst` で高レベル命令を低レベル命令へ分解する。
- `ir::static_condition_pruning` で分岐を静的簡約する。
- 外部 pass をパイプラインへ接続する。


まず、実行パスと入力ファイルを設定します。
`exe_path` と `GRIDSYNTH_PATH` は環境に合わせて変更してください。
また、Qurationに添付されているgridsynthを利用する場合は、必要に応じて実行権限を付与してください。

In [ ]:
import pathlib
import platform

import graphviz
from IPython.display import Code
import os

project_root = pathlib.Path("../../../..").resolve()
qret_path = project_root / "build" / "main"
if platform.system() == "Darwin": 
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth_macos"
else:
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth"

data_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_3.json"

os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

`qret` と `gridsynth` が実行可能で、入力 JSON を読み込めることを先に確認します。

In [ ]:
!qret --version

In [ ]:
!{gridsynth_path} --help

In [ ]:
Code(filename=data_path, language="JSON")

次に `opt` コマンドのオプションを確認します。

`qret opt` の実行形態は次の 2 つです。

- **引数直接指定**: `--pass` を複数回指定し、1 回の実行で順に適用します。
- **パイプラインファイル**: YAML に順序と入出力を定義して実行します。

まずは `--help` で利用可能なオプションを確認します。


In [ ]:
!qret opt --help

## 最適化対象の確認

まず `print -s` で対象モジュールの関数一覧を確認し、最適化対象を `Tutorial3Function` に決めます。

In [ ]:
!qret diagram -i {data_path} --function "Tutorial3Function" --graph-format "CallGraph" --display_num_calls -o { output_dir / "tutorial_3_diagram_call_graph.dot"}

graphviz.Source.from_file(output_dir / "tutorial_3_diagram_call_graph.dot")

In [ ]:
!qret print -i {data_path}

次に `Tutorial3Function` の未最適化状態を表示し、`Call` 命令の並びを確認します。

In [ ]:
!qret print -i {data_path} -f "Tutorial3Function"

続いて `-d 2` で呼び出し先まで展開し、展開前後の差分を確認します。

In [ ]:
!qret print -i {data_path} -f "Tutorial3Function" -d 2

## Quration に実装済みの pass を適用する場合

ここでは組み込み pass を順に適用し、`print` の出力差分を確認します。
手順は `ir::inliner` -> `ir::recursive_inliner` -> `ir::decompose_inst` -> `ir::static_condition_pruning` です。


`--pass` で直接呼び出せる主な組み込み pass は次のとおりです。

- `ir::inliner`: `Call` を 1 段だけ展開
- `ir::recursive_inliner`: `Call` を再帰的に展開
- `ir::decompose_inst`: 高レベル命令を低レベル命令へ分解
- `ir::static_condition_pruning`: 静的に確定できる分岐を簡約
- `ir::ignore_global_phase`: `GlobalPhase` を除去
- `ir::delete_consecutive_same_pauli`: 同一 Pauli の連続適用を除去
- `ir::delete_opt_hint`: 最適化ヒント命令を除去
- `ir::external`: 外部コマンドを pass として接続


### インライン展開（`Call` 命令の一段展開）

`ir::inliner` は、指定関数内の `Call` を 1 段だけインライン化します。
まずはこの pass 単体で、非再帰展開の結果を確認します。


In [ ]:
!qret opt -i {data_path} -o { output_dir / "tutorial_3_inlined.json"} -f "Tutorial3Function" --pass "ir::inliner"
!qret print -i  { output_dir / "tutorial_3_inlined.json"} -f "Tutorial3Function" -d 2

### 再帰的インライン展開（`Call` の再帰的置換）

`ir::recursive_inliner` は、展開後に現れた `Call` も続けて展開します。
`ir::inliner` との違いは、呼び出しチェーン全体を潰せる点です。


In [ ]:
!qret opt -i {data_path} -o { output_dir / "tutorial_3_recursively_inlined.json"} -f "Tutorial3Function" --pass "ir::recursive_inliner"
!qret print -i { output_dir / "tutorial_3_recursively_inlined.json"} -f "Tutorial3Function" -d 2

### 回転系命令の分解

`ir::decompose_inst` は、高レベル命令を低レベル命令列へ分解します。
この pass では `GRIDSYNTH_PATH` の設定が必要です。


In [ ]:
!qret opt -i {data_path} -o { output_dir / "tutorial_3_decomposed.json"} -f "Tutorial3Function" --pass "ir::decompose_inst"
!qret print -i { output_dir / "tutorial_3_decomposed.json"} -f "Tutorial3Function" -d 3

## 条件分岐の静的簡約
`ir::static_condition_pruning` は、到達時点で確定できる `Branch` / `Switch` を簡約し、到達不能経路を削除します。
本節では `tutorial_2.json` の `DiscreteDistribution` + `Switch` 構造で挙動を確認します。


`tutorial_2.json` の `Tutorial2Function` では、`Switch` の `registers` が `[@r0, @r1]` の順で評価されます。
`r0` が LSB として扱われるため、値は `value = 1*r0 + 2*r1` です。
この対応で `case` / `default` の選択先を読み取れます。


In [ ]:
tutorial_2_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_2.json"
!qret print -i {tutorial_2_path} -f "Tutorial2Function"

In [ ]:
!qret opt -i {tutorial_2_path} -o { output_dir / "tutorial_2_pruned.json"} -f "Tutorial2Function" --pass "ir::static_condition_pruning" --ir-static-condition-pruning-seed 0
!qret print -i {output_dir / "tutorial_2_pruned.json"} -f "Tutorial2Function" -d 2

`ir-static-condition-pruning-seed` を指定すると、`DiscreteDistribution` のサンプリング結果を再現できます。
同じ seed を使えば同じ分岐に収束するため、比較実験がしやすくなります。


seed を変えて実行し、残る `case` が切り替わることを確認します。
再現性のある比較を行う場合は、使用した seed を固定して記録します。


In [ ]:
!qret opt -i {tutorial_2_path} -o { output_dir / "tutorial_2_pruned_seed3.json"} -f "Tutorial2Function" --pass "ir::static_condition_pruning" --ir-static-condition-pruning-seed 3
!qret print -i { output_dir / "tutorial_2_pruned_seed3.json"} -f "Tutorial2Function" -d 2
!qret opt -i {tutorial_2_path} -o { output_dir / "tutorial_2_pruned_seed2.json"} -f "Tutorial2Function" --pass "ir::static_condition_pruning" --ir-static-condition-pruning-seed 2
!qret print -i { output_dir / "tutorial_2_pruned_seed2.json"} -f "Tutorial2Function" -d 2

ここまでで 4 つの pass を適用し、`print` 出力の差分を確認しました。

- `inliner` / `recursive_inliner`: `Call` 展開の粒度を制御
- `decompose_inst`: 命令列を低レベル化
- `static_condition_pruning`: 分岐と到達不能経路を簡約


## 外部実装の pass を連結する場合

外部実装を使う場合は、YAML パイプラインで `qret` pass と外部コマンドを連結します。
この例では `ir::inliner` の後に Python スクリプトを実行し、`Tutorial3Function` のみ更新します。


In [ ]:
tutorial_3_decompose_using_external_pass_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_3_decompose_using_external_pass.yaml"
Code(filename=tutorial_3_decompose_using_external_pass_path, language="YAML")

`data/decompose.py` は `CCX` を `H` / `CX` / `T` / `TDag` へ分解するサンプルです。
入出力 JSON の形式を確認してから実行します。


In [ ]:
tutorial_3_decompose_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_3_decompose.py"
Code(filename=tutorial_3_decompose_path, language="Python")

最後に `qret opt --pipeline` を実行し、`decomposed.json` の更新結果を確認します。

In [ ]:
!qret opt  --pipeline {tutorial_3_decompose_using_external_pass_path}
!qret print -i {output_dir / "tutorial_3_decomposed.json"} -f "Tutorial3Function" -d 3

本章では、`qret opt` で pass を適用する基本フローを確認しました。

- 先に `print` でベースラインを取る
- pass の目的に合わせて適用順を決める
- 必要に応じて外部 pass をパイプラインへ組み込む
- 実行後に `print` で差分を再確認する
